In [ ]:
import os
import numpy as np
import imageio.v3 as imageio
import torch
import torch.nn.functional as F


def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def save_u8(path, img01):
    img8 = (np.clip(img01, 0.0, 1.0) * 255.0 + 0.5).astype(np.uint8)
    imageio.imwrite(path, img8)


def fftshift(x):
    for dim in (-2, -1):
        n = x.size(dim)
        x = torch.roll(x, shifts=(n // 2,), dims=(dim,))
    return x


def ifftshift(x):
    for dim in (-2, -1):
        n = x.size(dim)
        x = torch.roll(x, shifts=(-(n // 2),), dims=(dim,))
    return x


def create_grid_like_numpy(num_pixels_x, dx, num_pixels_y=None, dy=None, device=None):
    if num_pixels_y is None:
        num_pixels_y = num_pixels_x

    if dy is None:
        dy = dx

    width = dx * num_pixels_x
    height = dy * num_pixels_y

    x = torch.linspace(-width / 2, width / 2, num_pixels_x, device=device, dtype=torch.float32)
    y = torch.linspace(-height / 2, height / 2, num_pixels_y, device=device, dtype=torch.float32)

    X, Y = torch.meshgrid(x, y, indexing="xy")

    return X, Y


def crop_center_torch(a, crop_h, crop_w):
    H, W = a.shape
    sx = W // 2 - crop_w // 2
    sy = H // 2 - crop_h // 2
    return a[sy:sy + crop_h, sx:sx + crop_w]


def crop_valid(a, border):
    if border <= 0:
        return a
    return a[border:-border, border:-border]


def precompute_angular_spectrum_kernel(ny, nx, dx, wavelength, z, device):
    def qidx(n):
        idx = torch.arange(-n / 2, n / 2, device=device, dtype=torch.float32)
        return (wavelength / dx / n * idx) ** 2

    qx = qidx(nx)
    qy = qidx(ny)[:, None]
    q = qy + qx[None, :]

    if dx < wavelength / np.sqrt(2):
        cond = q > 1.0
        q2 = torch.clamp(q, max=1.0)
        H = torch.exp(2j * np.pi / wavelength * z * torch.sqrt(1.0 - q2))
        H[cond] = 0.0
    else:
        H = torch.exp(2j * np.pi / wavelength * z * torch.sqrt(1.0 - q))

    return ifftshift(H).to(torch.complex64)


def create_circular_mask_torch(nx, dx, radius, ny=None, dy=None, device=None):
    if ny is None:
        ny = nx

    if dy is None:
        dy = dx

    X, Y = create_grid_like_numpy(nx, dx, ny, dy, device=device)
    R = torch.sqrt(X ** 2 + Y ** 2)

    S = torch.zeros((ny, nx), dtype=torch.float32, device=device)
    S[R < radius] = 1.0

    return S


def normalize01_detached(x):
    return x / (x.detach().abs().max() + 1e-12)


def compute_complex_coherence_factor_torch(source_intensity):
    g = fftshift(torch.fft.fft2(ifftshift(source_intensity)))
    return normalize01_detached(g)


def compute_schells_propagation_torch(Ic, gamma):
    tmp = fftshift(torch.fft.ifft2(ifftshift(Ic))) * gamma
    out = fftshift(torch.fft.fft2(ifftshift(tmp)))
    return torch.abs(out)


def gaussian_kernel_1d(window_size, sigma, device):
    coords = torch.arange(window_size, device=device, dtype=torch.float32) - (window_size - 1) / 2.0
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = g / (g.sum() + 1e-12)

    return g


def gaussian_window_2d(window_size, sigma, device):
    g1 = gaussian_kernel_1d(window_size, sigma, device)
    w2 = torch.outer(g1, g1)
    w2 = w2 / (w2.sum() + 1e-12)

    return w2[None, None, :, :]


def ssim_torch(img1, img2, window_size=11, sigma=1.5, data_range=1.0):
    x = img1[None, None, :, :]
    y = img2[None, None, :, :]
    w = gaussian_window_2d(window_size, sigma, img1.device)

    mu_x = F.conv2d(x, w, padding=window_size // 2)
    mu_y = F.conv2d(y, w, padding=window_size // 2)

    mu_x2 = mu_x * mu_x
    mu_y2 = mu_y * mu_y
    mu_xy = mu_x * mu_y

    sigma_x2 = F.conv2d(x * x, w, padding=window_size // 2) - mu_x2
    sigma_y2 = F.conv2d(y * y, w, padding=window_size // 2) - mu_y2
    sigma_xy = F.conv2d(x * y, w, padding=window_size // 2) - mu_xy

    C1 = (0.01 * data_range) ** 2
    C2 = (0.03 * data_range) ** 2

    num = (2 * mu_xy + C1) * (2 * sigma_xy + C2)
    den = (mu_x2 + mu_y2 + C1) * (sigma_x2 + sigma_y2 + C2)

    return (num / (den + 1e-12)).mean()


def make_soft_support_mask(num_px, border, taper_px, device):
    yy = torch.arange(num_px, device=device, dtype=torch.float32)[:, None]
    xx = torch.arange(num_px, device=device, dtype=torch.float32)[None, :]

    dist_left = xx
    dist_right = (num_px - 1) - xx
    dist_top = yy
    dist_bottom = (num_px - 1) - yy

    dist_edge = torch.minimum(
        torch.minimum(dist_left, dist_right),
        torch.minimum(dist_top, dist_bottom),
    )

    if taper_px <= 0:
        return (dist_edge >= border).to(torch.float32)

    t = (dist_edge - float(border)) / float(taper_px)
    t = torch.clamp(t, 0.0, 1.0)

    return 0.5 - 0.5 * torch.cos(np.pi * t)


def tv_huber_real(x, eps=1e-6):
    dx = x[:, 1:] - x[:, :-1]
    dy = x[1:, :] - x[:-1, :]

    tv_x = torch.mean(torch.sqrt(dx * dx + eps))
    tv_y = torch.mean(torch.sqrt(dy * dy + eps))

    return tv_x + tv_y


def double_well_amplitude(x):
    return torch.mean((x * (1.0 - x)) ** 2)


def background_white_loss_amp(x, background_mask):
    denom = torch.sum(background_mask) + 1e-12
    return torch.sum(background_mask * (x - 1.0) ** 2) / denom


def foreground_sparsity_amp(x, support_mask, eps=1e-6):
    denom = torch.sum(support_mask) + 1e-12
    return torch.sum(support_mask * torch.sqrt((1.0 - x) ** 2 + eps)) / denom


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

wavelength = ## define wavelength in meters ##
z_source_object = ## define source to object distance in meters ##
z_obj_cam = ## define object to camera distance in meters ##
dx_camera = ## define camera pixel size in meters ##

num_px = ## define number of pixels for reconstruction ##
pad_factor = ## define padding factor (e.g., 1.0 for no padding, 2.0 for double size) ##
pad = pad_factor * num_px
zpad = num_px + 2 * pad
ny = nx = zpad

source_radii = ## [define list of source radii in meters] ##

coh_names = ## [define list of coherence names corresponding to source_radii] ##

meas_root = ## "REPLACE_WITH_AMPLITUDE_MEASUREMENT_FOLDER" ##
base_recon_dir = ## "REPLACE_WITH_OUTPUT_FOLDER" ##

object_groups = ## [define list of object classes] ##

object_ids = ## [define list of object IDs] ##

reconstruct_all_objects = True
n_random_objects_per_class = 5
random_seed = 139

measurement_filename = ## "REPLACE_WITH_MEASUREMENT_FILENAME" ##

n_iters = ## define number of iterations for optimization ##
lr = ## define learning rate for optimization ##

roi_border = ## define border size to crop valid region from reconstruction ##

support_border = ## define border size for support mask ##
support_taper_px = ## define taper size in pixels for support mask ##

use_ssim_loss = True
w_huber = ## define weight for Huber loss ##
w_ssim = ## define weight for SSIM loss ##

lambda_tv_amp = ## define weight for total variation regularization ##
lambda_bg_amp = ## define weight for background regularization ##
lambda_sparse_amp = ## define weight for foreground sparsity regularization ##
lambda_dw_amp = ## define weight for double well regularization ##
huber_beta = ## define beta parameter for Huber loss ##

ensure_dir(base_recon_dir)

rng = np.random.default_rng(random_seed)

selected_object_ids_by_group = {}

for group_name in object_groups:
    if reconstruct_all_objects:
        selected_ids = list(object_ids)
    else:
        n_select = min(n_random_objects_per_class, len(object_ids))
        selected_ids = list(rng.choice(object_ids, size=n_select, replace=False))
        selected_ids = sorted(selected_ids, key=lambda x: int(x))

    selected_object_ids_by_group[group_name] = selected_ids

print("Selected object IDs by class:")

for group_name, selected_ids in selected_object_ids_by_group.items():
    print(f"{group_name}: {selected_ids}")

print("Precomputing fixed forward terms.")

X, Y = create_grid_like_numpy(nx, dx_camera, ny, dx_camera, device=device)

phase_factor = torch.exp(
    1j * np.pi / (wavelength * z_source_object) * (X ** 2 + Y ** 2)
).to(torch.complex64)

H_as = precompute_angular_spectrum_kernel(
    ny,
    nx,
    dx_camera,
    wavelength,
    z_obj_cam,
    device=device,
)

dx_src = dx_camera * z_source_object / z_obj_cam
dy_src = dx_src

support_mask = make_soft_support_mask(
    num_px=num_px,
    border=support_border,
    taper_px=support_taper_px,
    device=device,
)

background_mask = 1.0 - support_mask


def apply_support(v):
    v01 = v.clamp(0.0, 1.0)
    return support_mask * v01 + background_mask * 1.0


def gvs_forward_amp_only_normalized(amp_2000, gamma):
    amp = amp_2000.clamp(0.0, 1.0)
    field_pad = F.pad(amp, (pad, pad, pad, pad), mode="constant", value=1.0).to(torch.complex64)
    Uc = torch.fft.ifft2(torch.fft.fft2(field_pad * phase_factor) * H_as)
    Ic = (torch.abs(Uc) ** 2).to(torch.float32)
    Igvs_full = compute_schells_propagation_torch(Ic, gamma)
    I_crop = crop_center_torch(Igvs_full, num_px, num_px).to(torch.float32)

    return normalize01_detached(I_crop)


def coherent_forward_amp_only_normalized(amp_2000):
    amp = amp_2000.clamp(0.0, 1.0)
    field_pad = F.pad(amp, (pad, pad, pad, pad), mode="constant", value=1.0).to(torch.complex64)
    Uc = torch.fft.ifft2(torch.fft.fft2(field_pad * phase_factor) * H_as)
    Ic = (torch.abs(Uc) ** 2).to(torch.float32)
    I_crop = crop_center_torch(Ic, num_px, num_px).to(torch.float32)

    return normalize01_detached(I_crop)


try:
    for obj_group in object_groups:
        for obj_id in selected_object_ids_by_group[obj_group]:
            print("=" * 100)
            print(f"OBJECT: {obj_group}/{obj_id}")
            print("=" * 100)

            for coh_name, radius in zip(coh_names, source_radii):
                meas_path = os.path.join(
                    meas_root,
                    obj_group,
                    obj_id,
                    coh_name,
                    measurement_filename,
                )

                if not os.path.exists(meas_path):
                    print(f"Missing measurement: {meas_path}")
                    continue

                I_np = imageio.imread(meas_path, mode="L").astype(np.float32) / 255.0
                I_np = I_np / max(1e-8, float(I_np.max()))
                I_meas = torch.from_numpy(I_np).to(device)

                print("-" * 100)
                print(f"MEASUREMENT: {obj_group}/{obj_id}/{coh_name}   source_radius = {radius * 1e6:.1f} um")
                print("-" * 100)

                out_dir = os.path.join(
                    base_recon_dir,
                    obj_group,
                    obj_id,
                    coh_name,
                )

                ensure_dir(out_dir)

                src_I = create_circular_mask_torch(
                    nx,
                    dx_src,
                    radius,
                    ny,
                    dy_src,
                    device=device,
                )

                gamma = compute_complex_coherence_factor_torch(src_I).to(torch.complex64)

                del src_I

                v = torch.full(
                    (num_px, num_px),
                    0.5,
                    device=device,
                    dtype=torch.float32,
                ).requires_grad_(True)

                opt = torch.optim.Adam([v], lr=lr)

                for it in range(1, n_iters + 1):
                    opt.zero_grad()

                    amp = apply_support(v)
                    I_pred = gvs_forward_amp_only_normalized(amp, gamma)

                    huber = F.smooth_l1_loss(
                        I_pred,
                        I_meas,
                        beta=huber_beta,
                        reduction="mean",
                    )

                    reg_tv = tv_huber_real(amp)
                    reg_bg = background_white_loss_amp(amp, background_mask)
                    reg_dw = double_well_amplitude(amp)
                    reg_sparse = foreground_sparsity_amp(amp, support_mask)

                    if use_ssim_loss:
                        ssim_val = ssim_torch(
                            I_pred,
                            I_meas,
                            window_size=11,
                            sigma=1.5,
                            data_range=1.0,
                        )

                        loss = (
                            w_huber * huber
                            + w_ssim * (1.0 - ssim_val)
                            + lambda_tv_amp * reg_tv
                            + lambda_bg_amp * reg_bg
                            + lambda_sparse_amp * reg_sparse
                            + lambda_dw_amp * reg_dw
                        )
                    else:
                        loss = (
                            w_huber * huber
                            + lambda_tv_amp * reg_tv
                            + lambda_bg_amp * reg_bg
                            + lambda_sparse_amp * reg_sparse
                            + lambda_dw_amp * reg_dw
                        )

                    loss.backward()
                    opt.step()

                    with torch.no_grad():
                        v.clamp_(0.0, 1.0)

                    if it == 1 or it % 20 == 0:
                        print(f"[GVS] Iter {it:04d}/{n_iters:04d}  Loss={float(loss.item()):.6e}")

                with torch.no_grad():
                    amp_gvs = apply_support(v)

                v2 = torch.full(
                    (num_px, num_px),
                    0.5,
                    device=device,
                    dtype=torch.float32,
                ).requires_grad_(True)

                opt2 = torch.optim.Adam([v2], lr=lr)

                for it in range(1, n_iters + 1):
                    opt2.zero_grad()

                    amp_c = apply_support(v2)
                    I_pred_c = coherent_forward_amp_only_normalized(amp_c)

                    huber = F.smooth_l1_loss(
                        I_pred_c,
                        I_meas,
                        beta=huber_beta,
                        reduction="mean",
                    )

                    reg_tv = tv_huber_real(amp_c)
                    reg_bg = background_white_loss_amp(amp_c, background_mask)
                    reg_dw = double_well_amplitude(amp_c)
                    reg_sparse = foreground_sparsity_amp(amp_c, support_mask)

                    if use_ssim_loss:
                        ssim_val = ssim_torch(
                            I_pred_c,
                            I_meas,
                            window_size=11,
                            sigma=1.5,
                            data_range=1.0,
                        )

                        loss = (
                            w_huber * huber
                            + w_ssim * (1.0 - ssim_val)
                            + lambda_tv_amp * reg_tv
                            + lambda_bg_amp * reg_bg
                            + lambda_sparse_amp * reg_sparse
                            + lambda_dw_amp * reg_dw
                        )
                    else:
                        loss = (
                            w_huber * huber
                            + lambda_tv_amp * reg_tv
                            + lambda_bg_amp * reg_bg
                            + lambda_sparse_amp * reg_sparse
                            + lambda_dw_amp * reg_dw
                        )

                    loss.backward()
                    opt2.step()

                    with torch.no_grad():
                        v2.clamp_(0.0, 1.0)

                    if it == 1 or it % 20 == 0:
                        print(f"[COH] Iter {it:04d}/{n_iters:04d}  Loss={float(loss.item()):.6e}")

                with torch.no_grad():
                    amp_coh = apply_support(v2)

                amp_gvs_crop = crop_valid(amp_gvs, roi_border).detach().cpu().numpy()
                amp_coh_crop = crop_valid(amp_coh, roi_border).detach().cpu().numpy()

                save_u8(os.path.join(out_dir, "amp_GVS.png"), amp_gvs_crop)
                save_u8(os.path.join(out_dir, "amp_COH.png"), amp_coh_crop)

                print(f"Saved reconstructions to: {out_dir}")

                del gamma
                del v, v2
                del amp_gvs, amp_coh
                del I_meas

                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

finally:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

print("Done. Reconstructions saved under:", os.path.abspath(base_recon_dir))
